# Clase 8 - kNN y SVM

Programamos kNN desde cero para ver que no tiene magia, lo usamos para
reconocer dígitos escritos a mano (incluso dibujados por ustedes) y
después pasamos a SVM viendo márgenes, vectores de soporte y el truco del kernel.

## 1. kNN desde cero: ¿te va a gustar esta canción?

Spotify describe cada canción con números como la energía y la
bailabilidad (de 0 a 1). Vamos a simular 40 canciones que alguien escuchó y si
le gustaron o no.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=0)

# Le gustan las canciones con mucha energía y bailables, no le gustan las tranquilas
gustan = rng.normal([0.75, 0.70], 0.12, size=(20, 2))
no_gustan = rng.normal([0.35, 0.40], 0.15, size=(20, 2))

X_canciones = np.clip(np.vstack([gustan, no_gustan]), 0, 1)
y_canciones = np.array([1] * 20 + [0] * 20)       # 1 = le gustó

def graficar_canciones(nueva=None):
    plt.scatter(*X_canciones[y_canciones == 1].T, c="tab:green", label="le gustó")
    plt.scatter(*X_canciones[y_canciones == 0].T, c="tab:red", label="no le gustó")
    if nueva is not None:
        plt.scatter(*nueva, c="black", marker="*", s=300, label="canción nueva")
    plt.xlabel("Energía")
    plt.ylabel("Bailabilidad")
    plt.legend()

cancion_nueva = np.array([0.50, 0.70])
graficar_canciones(cancion_nueva)
plt.show()

¿Le va a gustar la canción nueva? kNN responde así:

1. calcular la distancia de la canción nueva a todas las conocidas,
2. quedarse con las `k` más cercanas,
3. votar.

In [ ]:
def knn_predecir(x_nuevo, X, y, k):
    distancias = np.sqrt(np.sum((X - x_nuevo) ** 2, axis=1))   # distancia euclídea a cada punto
    vecinos = np.argsort(distancias)[:k]                       # índices de los k más cercanos
    votos = np.bincount(y[vecinos])                            # cuántos votos tiene cada clase
    return votos.argmax(), vecinos

for k in [1, 3, 7]:
    prediccion, vecinos = knn_predecir(cancion_nueva, X_canciones, y_canciones, k)
    print(f"k={k}: votos de los vecinos = {y_canciones[vecinos]} -> predicción: {prediccion}")

In [ ]:
k = 7
prediccion, vecinos = knn_predecir(cancion_nueva, X_canciones, y_canciones, k)

graficar_canciones(cancion_nueva)
for v in vecinos:
    plt.plot(*zip(cancion_nueva, X_canciones[v]), color="gray", linestyle="--", linewidth=1)
plt.title(f"Los {k} vecinos más cercanos")
plt.show()

Con `k=1` y `k=3` el modelo dice que le va a gustar, y con `k=7` dice que
no. La canción está justo en la zona donde se mezclan las dos clases, y la
respuesta depende de cuántos vecinos consultemos.

Comprobamos que scikit-learn hace exactamente lo mismo:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in [1, 3, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_canciones, y_canciones)
    print(f"k={k}: scikit-learn predice {knn.predict([cancion_nueva])[0]}")

Fíjense que `fit` en kNN no aprende nada: solo guarda los datos. Todo el
trabajo pasa en `predict`!.

## 2. ¿Qué `k` elegimos?

Volvamos a las medialunas ruidosas de la clase de árboles para ver cómo
cambia la frontera de decisión según `k`.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.inspection import DecisionBoundaryDisplay

X, y = make_moons(n_samples=400, noise=0.35, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

def graficar_frontera(modelo, ax, titulo):
    DecisionBoundaryDisplay.from_estimator(modelo, X, ax=ax, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", edgecolor="k", s=20)
    ax.set_title(
        f"{titulo}\ntrain: {modelo.score(X_train, y_train):.2f} | "
        f"test: {modelo.score(X_test, y_test):.2f}"
    )

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axes, [1, 15, 150]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    graficar_frontera(knn, ax, f"k={k}")
plt.show()

In [ ]:
valores_k = range(1, 101, 2)
acc_train, acc_test = [], []

for k in valores_k:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc_train.append(knn.score(X_train, y_train))
    acc_test.append(knn.score(X_test, y_test))

plt.plot(valores_k, acc_train, label="train")
plt.plot(valores_k, acc_test, label="test")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

Es lo mismo que con `max_depth` en los árboles, pero al revés:
con `k=1` el modelo memoriza (100% en train), y con `k` promedia tanto que pierde la forma de las medialunas.

## 3. ¿Por qué hay que escalar?

Simulamos un banco que decide si aprueba un crédito según la edad (en
años) y el ingreso mensual (en pesos). Las dos variables importan.

In [ ]:
n = 600
edad = rng.uniform(18, 70, n)
ingreso = rng.uniform(300_000, 3_000_000, n)

puntaje = 0.08 * (edad - 40) + (ingreso - 1_500_000) / 600_000 + rng.normal(0, 0.5, n)
aprobado = (puntaje > 0).astype(int)

creditos = pd.DataFrame({"edad": edad, "ingreso": ingreso, "aprobado": aprobado})
creditos.describe().round(1)

In [ ]:
from sklearn.preprocessing import StandardScaler

X = creditos[["edad", "ingreso"]]
y = creditos["aprobado"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("Sin escalar:", round(knn.score(X_test, y_test), 3))

scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_esc, y_train)
print("Escalado:   ", round(knn.score(X_test_esc, y_test), 3))

¿Por qué tanta diferencia? Pensemos en dos clientes: uno de 20 años y otro
de 65, los dos con un ingreso de \$1.000.000. Y otro par: dos clientes de la
misma edad, con ingresos de \$1.000.000 y \$1.001.000.

In [ ]:
a = np.array([20, 1_000_000])
b = np.array([65, 1_000_000])
c = np.array([40, 1_001_000])
d = np.array([40, 1_000_000])

print("Distancia entre 20 y 65 años, mismo ingreso:      ", np.linalg.norm(a - b))
print("Distancia entre misma edad, $1000 de diferencia:  ", np.linalg.norm(c - d))

Sin escalar, una diferencia de \$1.000 pesa más que 45 años de diferencia.
La edad prácticamente no cuenta en la distancia. Después de escalar, las dos
variables quedan en unidades comparables (desvíos estándar).

## 4. Reconociendo dígitos escritos a mano

scikit-learn trae unas 1800 imágenes de dígitos escritos a mano, de 8×8
píxeles. Cada píxel es un número de 0 (blanco) a 16 (negro), así que cada
imagen es un punto en un espacio de 64 dimensiones. kNN funciona igual
que con 2 dimensiones: busca las imágenes más parecidas.

In [ ]:
from sklearn.datasets import load_digits

digitos = load_digits()
print(digitos.images.shape)   # 1797 imágenes de 8x8
print(digitos.data.shape)     # las mismas imágenes aplanadas en 64 columnas

fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for ax, imagen, etiqueta in zip(axes.flat, digitos.images, digitos.target):
    ax.imshow(imagen, cmap="gray_r")
    ax.set_title(etiqueta)
    ax.axis("off")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    digitos.data, digitos.target, test_size=0.25, random_state=0
)

knn_digitos = KNeighborsClassifier(n_neighbors=3)
knn_digitos.fit(X_train, y_train)

print("Accuracy en test:", round(knn_digitos.score(X_test, y_test), 3))

Veamos los errores

In [ ]:
predicciones = knn_digitos.predict(X_test)
errores = np.where(predicciones != y_test)[0]
print("Cantidad de errores:", len(errores), "de", len(y_test))

fig, axes = plt.subplots(1, len(errores), figsize=(1.5 * len(errores), 2))
for ax, i in zip(np.atleast_1d(axes), errores):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"real {y_test[i]}\npred {predicciones[i]}", fontsize=9)
    ax.axis("off")
plt.show()

Algunos errores son entendibles hasta para
una persona.

### Dibujen su propio dígito

Editen el dibujo de abajo usando `#` para los píxeles oscuros, `+` para los
grises y espacios para el blanco (8 filas de 8 caracteres). El modelo va a
decir qué número ve, y nos va a mostrar los vecinos que usó para decidir.

In [ ]:
mi_digito = [
    "  ######",
    "  ######",
    "     +##",
    "    +## ",
    "   ###+ ",
    "   ##   ",
    "  +##   ",
    "  ##    ",
]

valores = {" ": 0, "+": 8, "#": 16}
pixeles = np.array([[valores[c] for c in fila.ljust(8)[:8]] for fila in mi_digito])

prediccion = knn_digitos.predict(pixeles.reshape(1, -1))[0]
_, vecinos = knn_digitos.kneighbors(pixeles.reshape(1, -1), n_neighbors=5)

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
axes[0].imshow(pixeles, cmap="gray_r")
axes[0].set_title(f"tu dibujo\n-> {prediccion}")
for ax, v in zip(axes[1:], vecinos[0]):
    ax.imshow(X_train[v].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"vecino: {y_train[v]}")
for ax in axes:
    ax.axis("off")
plt.show()

## 5. SVM: la calle más ancha

Empezamos con dos grupos que se pueden separar con una recta. Usamos un `C`
muy grande para que SVM no tolere ningún punto dentro de la calle.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X, y = make_blobs(n_samples=60, centers=2, cluster_std=1.0, random_state=6)

svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

def graficar_svm(modelo, X, y, ax=None, titulo=""):
    ax = ax or plt.gca()
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=30)
    DecisionBoundaryDisplay.from_estimator(
        modelo, X, ax=ax, plot_method="contour", response_method="decision_function",
        levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"],
    )
    ax.scatter(*modelo.support_vectors_.T, s=150, facecolors="none", edgecolors="k", linewidths=1.5)
    ax.set_title(titulo)

graficar_svm(svm, X, y, titulo=f"Vectores de soporte: {len(svm.support_vectors_)}")
plt.show()

- La línea continua es la frontera de decisión.
- Las líneas punteadas son el margen.
- Los puntos con círculo son los vectores de soporte.

### Margen blando: el efecto de `C`

Ahora con grupos que se pisan un poco, como pasa con datos reales.

In [ ]:
X, y = make_blobs(n_samples=100, centers=2, cluster_std=2.2, random_state=6)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, C in zip(axes, [0.01, 1, 100]):
    svm = SVC(kernel="linear", C=C)
    svm.fit(X, y)
    graficar_svm(svm, X, y, ax=ax,
                 titulo=f"C={C} | vectores de soporte: {len(svm.support_vectors_)}")
plt.show()

Con `C` chico la calle es ancha y muchos puntos quedan adentro (todos son
vectores de soporte). Con `C` grande la calle se angosta para cometer la
menor cantidad de errores posible en train.

## 6. El truco del kernel

¿Qué pasa si una clase está rodeada por la otra? Ninguna recta puede
separarlas.

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=300, factor=0.4, noise=0.08, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, kernel in zip(axes, ["linear", "rbf"]):
    svm = SVC(kernel=kernel)
    svm.fit(X, y)
    DecisionBoundaryDisplay.from_estimator(svm, X, ax=ax, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=20)
    ax.set_title(f"kernel='{kernel}' | accuracy: {svm.score(X, y):.2f}")
plt.show()

El kernel lineal no puede hacer nada mejor que tirar la moneda. El kernel
`rbf` encierra al grupo del medio sin problemas.

¿Cómo lo logra? La idea es agregar una dimensión nueva donde los datos
sí se puedan separar con un plano. Por ejemplo, la distancia al centro al
cuadrado: $z = x_1^2 + x_2^2$. Veamos los mismos puntos en 3D:

In [ ]:
z = X[:, 0] ** 2 + X[:, 1] ** 2

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(X[:, 0], X[:, 1], z, c=y, cmap="coolwarm", edgecolor="k", s=20)

# un plano horizontal separa las dos clases
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 10), np.linspace(-1.2, 1.2, 10))
ax.plot_surface(xx, yy, np.full_like(xx, 0.5), alpha=0.25, color="green")

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("z = x1² + x2²")
ax.view_init(elev=15, azim=30)
plt.show()

En 3D, los puntos del centro quedan abajo y los de afuera arriba: un
plano los separa perfecto. El truco del kernel hace algo así, pero sin
calcular nunca las coordenadas nuevas de verdad, y con dimensiones extra
mucho más ricas que esta.

## 7. Duelo final: todos los clasificadores

Comparamos los cuatro modelos que vimos hasta ahora sobre tres datasets de
juguete con formas distintas. Para no repetir el `StandardScaler` a mano
usamos `make_pipeline`, que encadena el escalado y el modelo en un solo
objeto con `fit` y `predict`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

modelos = {
    "Logística": make_pipeline(StandardScaler(), LogisticRegression()),
    "Árbol (depth=5)": DecisionTreeClassifier(max_depth=5, random_state=0),
    "kNN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}

datasets = {
    "Medialunas": make_moons(n_samples=300, noise=0.3, random_state=0),
    "Círculos": make_circles(n_samples=300, factor=0.5, noise=0.15, random_state=0),
    "Manchas": make_blobs(n_samples=300, centers=2, cluster_std=2.0, random_state=6),
}

fig, axes = plt.subplots(len(datasets), len(modelos), figsize=(16, 11))
resultados = {}

for fila, (nombre_datos, (X, y)) in enumerate(datasets.items()):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    for col, (nombre_modelo, modelo) in enumerate(modelos.items()):
        modelo.fit(X_train, y_train)
        acc = modelo.score(X_test, y_test)
        resultados.setdefault(nombre_modelo, {})[nombre_datos] = round(acc, 2)

        ax = axes[fila, col]
        DecisionBoundaryDisplay.from_estimator(modelo, X, ax=ax, alpha=0.3, cmap="coolwarm")
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", s=15)
        ax.set_title(f"{nombre_modelo}\ntest: {acc:.2f}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(nombre_datos, fontsize=12)

plt.tight_layout()
plt.show()

pd.DataFrame(resultados)

Miren la forma de cada frontera! Recta en la logística, escalones en el
árbol, manchas suaves en kNN y curvas en SVM. ¿Cómo comparar modelos de forma
justa (y no confiar en un único split de test)? Es el tema de la próxima
clase.

## Ejercicios

**Ejercicio 1.** En el dataset de `creditos` (escalado), buscar el mejor `k`
probando todos los impares del 1 al 51 y graficar el accuracy de train y
test en función de `k`. ¿Qué `k` es mejor?

**Ejercicio 2.** Dibujar en `mi_digito` otros números. Encontrar al menos un
dibujo que ustedes reconozcan claramente pero el modelo clasifique mal.
¿Qué tienen en común los vecinos que eligió?

**Ejercicio 3.** Entrenar un `SVC(kernel="rbf")` sobre los dígitos y comparar
su accuracy en test contra kNN. Medir también cuánto tarda cada uno en
`fit` y en `predict` (pueden usar `time.time()` antes y después). ¿Se
cumple que kNN es rápido para entrenar y lento para predecir?

**Ejercicio 4.** El kernel `rbf` tiene otro hiperparámetro, `gamma`, que
controla qué tan "local" es la influencia de cada punto. Sobre las
medialunas, graficar la frontera de `SVC(kernel="rbf", gamma=g)` para `g` en
`[0.1, 1, 10, 100]` mostrando accuracy de train y test. ¿Con cuál hay
overfitting?

**Ejercicio 5.** Agregar un quinto modelo:
`RandomForestClassifier` (de `sklearn.ensemble`) y un cuarto dataset:
`make_moons(n_samples=300, noise=0.05, random_state=0)` (casi sin ruido).
¿Cambia cuál es el mejor modelo cuando los datos no tienen ruido?

In [ ]:
# Ejercicio 1: 

In [ ]:
# Ejercicio 2: 

In [ ]:
# Ejercicio 3: 

In [ ]:
# Ejercicio 4: 

In [ ]:
# Ejercicio 5: 